# 00 — Multilingual document intelligence: the measured system

**Question:** how should an open vision-language model retrieve and use evidence when compute and labeled data are limited?

This project compares three reader configurations, searches a broad multilingual retrieval feature space, runs the complete PDF-to-answer pipeline, and tests a fixed second read of the model's own citations. Every result is tied to source code, model revisions, data hashes, durable cloud artifacts, and executed notebooks.

Start with the verified overview below, then read **05** for the complete system, **03** for model and cost comparisons, and **04** for feature research. The saved outputs are available without AWS, Hugging Face, or a GPU.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.reporting import load_report
from lava.evaluation.system import load_summary
from lava.evaluation.walkthrough import (
    TABLE_STYLE,
    comparison_tables,
    render_table,
    training_rates,
)
from lava.notebook_support import find_repo_root
from lava.readers.refinement import load_refinement
from lava.readers.runtime_logging import RuntimeEventLogger
from lava.retrieval.pipeline import load_public_report

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.protocol")
with logger.stage("01_verify_benchmark_results", heartbeat_seconds=15):
    report = load_report(ROOT)
    retrieval = load_public_report(ROOT)
    pricing = json.loads((ROOT / "reports/aws/training_prices.json").read_text())
    tables = comparison_tables(report, training_rates(pricing))
    assert len(report["current_models"]) == 3
    assert all(row["Stage"] == "Full pilot scored" for row in tables["coverage"])
    assert report["expected_questions"] == retrieval["question_count"] == 16
    assert retrieval["reader_evaluated"] is False
    assert retrieval["local_lava_overall"] is None
    quality_columns = ("Reader", "Semantic VQA", "Evidence F1", "Local LAVA overall")
    display(
        HTML(
            TABLE_STYLE
            + render_table(
                [{key: row[key] for key in quality_columns} for row in tables["quality"]],
                percent_columns=quality_columns[1:],
                caption="Completed reader benchmark · correct evidence supplied",
            )
        )
    )
    retrieval_rows = []
    for method, label in (("page_order", "Page-order control"), ("bm25", "Multilingual BM25")):
        values = retrieval["methods"][method]["question_average"]["5"]
        retrieval_rows.append(
            {
                "Method": label,
                "Pages": 5,
                "Evidence recall": values["recall_at_k"],
                "All evidence found": values["all_evidence_at_k"],
            }
        )
    display(
        HTML(
            render_table(
                retrieval_rows,
                percent_columns=("Evidence recall", "All evidence found"),
                caption="Completed retrieval benchmark · five-page diagnostic",
            )
        )
    )
    system = load_summary(ROOT)
    refinement = load_refinement(ROOT)
    assert system is not None and refinement is not None
    display(
        HTML(
            render_table(
                [
                    {
                        "System condition": name,
                        "Answer credit": values["metrics"]["question_micro"]["answer"],
                        "Evidence F1": values["metrics"]["question_micro"]["grounding"],
                        "Local LAVA": values["metrics"]["question_micro"]["overall"],
                    }
                    for name, values in (
                        ("BM25 + 9B first pass", system),
                        ("9B reread of self-cited pages", refinement),
                    )
                ],
                percent_columns=("Answer credit", "Evidence F1", "Local LAVA"),
                caption="Actual retrieval-to-answer measurements; all 16 development questions",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.001, "event": "01_verify_benchmark_results.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-10T03:47:42.771+00:00"}


Reader,Semantic VQA,Evidence F1,Local LAVA overall
Qwen3.5 · 4B,50.62%,97.02%,73.82%
Qwen3.5 · 9B,80.15%,93.90%,87.02%
Qwen3.8 · 27B NF4,70.98%,89.73%,80.36%


Method,Pages,Evidence recall,All evidence found
Page-order control,5,45.31%,37.50%
Multilingual BM25,5,95.31%,87.50%


System condition,Answer credit,Evidence F1,Local LAVA
BM25 + 9B first pass,48.90%,86.25%,67.57%
9B reread of self-cited pages,67.65%,88.33%,77.99%


{"component": "notebook.protocol", "elapsed_seconds": 0.31, "event": "01_verify_benchmark_results.completed", "level": "INFO", "stage_elapsed_seconds": 0.31, "timestamp_utc": "2026-09-10T03:47:43.081+00:00"}


## 2. Interpret the findings

9B achieved the highest oracle-page local score among the three measured reader configurations. The first integrated run then exposed a substantial gap between evidence coverage and answering accuracy. A targeted second-read experiment tests context selection using the same 9B weights.

The feature audit generated 1,582 candidates, found 1,390 globally distinct full rankings, and retained the baseline under conservative document-isolated selection. The visual-only challenger underperformed; its hybrid was exploratory. The final conclusions come from measured comparisons and explicit limitations.

All labels are from 16 previously examined training questions across five PDFs. These results demonstrate an engineering workflow; they do not establish held-out accuracy, language-wide performance, organizer-server parity, or state of the art.

In [2]:
with logger.stage("02_verify_data", heartbeat_seconds=15):
    manifest = json.loads((ROOT / "reports/raw_data_manifest_summary.json").read_text())
    audit = json.loads((ROOT / "reports/data_audit/data_audit_summary_full.json").read_text())
    assert manifest["complete"] and manifest["verified_file_count"] == 208
    assert audit["audit_mode"] == "full" and audit["audited_pdf_count_for_mode"] == 205
    rows = []
    for profile in audit["csv_profiles"]:
        if profile["selected_columns"]["question"] is None:
            continue
        split = "Training" if profile["selected_columns"]["answer"] else "Test"
        rows.append(
            {
                "Split": split,
                "Questions": profile["row_count"],
                "PDFs": profile["referenced_document_count"],
                "Japanese": profile["language_counts"]["ja"],
                "Vietnamese": profile["language_counts"]["vi"],
            }
        )
    display(HTML(TABLE_STYLE + render_table(rows, caption="Data audit complete")))
    display(
        HTML(
            render_table(
                [
                    {"Check": "Verified raw files", "Result": manifest["verified_file_count"]},
                    {"Check": "PDFs audited", "Result": audit["audited_pdf_count_for_mode"]},
                    {
                        "Check": "Exact PDF duplicates across splits",
                        "Result": audit["cross_split_exact_duplicate_pdf_group_count"],
                    },
                ],
                caption="Checks completed before modeling",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.316, "event": "02_verify_data.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-10T03:47:43.086+00:00"}


Split,Questions,PDFs,Japanese,Vietnamese
Test,624,200,587,37
Training,16,5,15,1


Check,Result
Verified raw files,208
PDFs audited,205
Exact PDF duplicates across splits,0


{"component": "notebook.protocol", "elapsed_seconds": 0.318, "event": "02_verify_data.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-10T03:47:43.089+00:00"}


## 4. Review the completed deliverable

The public repository includes the full system evaluation, paired document and question diagnostics, measured cloud runtime, an executable feature-selection audit, checksum-verified recovery, and six canonical notebooks. Private PDFs and raw model outputs remain in the versioned experiment store. Public aggregates and execution manifests make the findings reviewable without accessing private data.

Kaggle submission and a hosted inference application remain separate optional operations. The research system and its evaluation are complete.

In [3]:
with logger.stage("03_verify_release_scope", heartbeat_seconds=15):
    display(
        HTML(
            render_table(
                [
                    {
                        "Deliverable": "Data verification",
                        "Evidence": "208 files and 205 PDFs audited",
                    },
                    {
                        "Deliverable": "Reader comparison",
                        "Evidence": "Three full 16-question pilots scored",
                    },
                    {"Deliverable": "Retrieval evaluation", "Evidence": retrieval["status"]},
                    {"Deliverable": "Integrated answering", "Evidence": system["status"]},
                    {"Deliverable": "Context refinement", "Evidence": refinement["status"]},
                    {
                        "Deliverable": "Model and systems analysis",
                        "Evidence": "Scores, slices, uncertainty, runtime, memory, cost",
                    },
                    {
                        "Deliverable": "Reproducible presentation",
                        "Evidence": "Six executed canonical notebooks with verified manifests",
                    },
                    {
                        "Deliverable": "Release scope",
                        "Evidence": "Measured end-to-end research system; submission optional",
                    },
                ],
                caption="Portfolio deliverables and their evidence",
            )
        )
    )
logger.emit(
    "protocol.walkthrough.completed",
    raw_files=manifest["verified_file_count"],
    complete_reader_pilots=len(report["current_models"]),
    retrieval_evaluated=retrieval["status"] == "Full-document retrieval evaluated",
    release_scope="measured_document_intelligence_system",
    submission_required=False,
)

{"component": "notebook.protocol", "elapsed_seconds": 0.323, "event": "03_verify_release_scope.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-10T03:47:43.094+00:00"}


Deliverable,Evidence
Data verification,208 files and 205 PDFs audited
Reader comparison,Three full 16-question pilots scored
Retrieval evaluation,Full-document retrieval evaluated
Integrated answering,Retrieved-evidence pilot scored
Context refinement,Self-citation refinement scored
Model and systems analysis,"Scores, slices, uncertainty, runtime, memory, cost"
Reproducible presentation,Six executed canonical notebooks with verified manifests
Release scope,Measured end-to-end research system; submission optional


{"component": "notebook.protocol", "elapsed_seconds": 0.326, "event": "03_verify_release_scope.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-10T03:47:43.096+00:00"}


{"complete_reader_pilots": 3, "component": "notebook.protocol", "elapsed_seconds": 0.327, "event": "protocol.walkthrough.completed", "level": "INFO", "raw_files": 208, "release_scope": "measured_document_intelligence_system", "retrieval_evaluated": true, "submission_required": false, "timestamp_utc": "2026-09-10T03:47:43.097+00:00"}


## 5. Follow the evidence

- [05 — End-to-end system evaluation](05_end_to_end_system_evaluation.ipynb): first pass, self-citation reread, errors, cost and recovery.
- [03 — Model quality and cost](03_model_scaling_and_cost.ipynb): 4B, 9B and quantized 27B reader comparisons.
- [04 — Evidence retrieval](04_evidence_retrieval.ipynb): multilingual features, every selection fold, visual challenger and resume evidence.
- [01 — Design](01_oracle_reader_benchmark_design.ipynb) and [02 — Execution](02_verified_gpu_execution.ipynb): detailed contracts and engineering decisions.